# Validate XEM2 Cross-Section Ratio Config

This notebook checks the first-pass YAML configuration extracted from `codex_xsec.ipynb`. It is intentionally lightweight: the goal is to verify that the analysis contract is parseable and that the major sections Casey suggested are present.

In [ ]:
from pathlib import Path
import json
import subprocess

config_path = Path('xem2_xsec_ratio.yaml')
assert config_path.exists(), f'Missing config file: {config_path.resolve()}'

def load_yaml(path):
    try:
        import yaml
        return yaml.safe_load(path.read_text())
    except ModuleNotFoundError:
        ruby = "require 'yaml'; require 'json'; puts YAML.load_file(ARGV[0]).to_json"
        result = subprocess.run(['ruby', '-e', ruby, str(path)], check = True, capture_output = True, text = True)
        return json.loads(result.stdout)

cfg = load_yaml(config_path)
list(cfg.keys())

In [ ]:
required_top_level = ['schema_version', 'analysis', 'run', 'paths', 'inputs', 'outputs', 'binning', 'branches', 'cuts', 'histograms', 'corrections', 'derived_columns', 'modules']
missing = [key for key in required_top_level if key not in cfg]
assert not missing, f'Missing required top-level sections: {missing}'

required_cuts = ['data_electron', 'simc_acceptance', 'final_fit_points']
missing_cuts = [key for key in required_cuts if key not in cfg['cuts']]
assert not missing_cuts, f'Missing cuts: {missing_cuts}'

required_modules = ['data_yield_extraction', 'simc_yield_extraction', 'born_cross_section_extraction', 'ratio_and_slope_fit']
missing_modules = [key for key in required_modules if key not in cfg['modules']]
assert not missing_modules, f'Missing modules: {missing_modules}'

print('Config validation passed.')

In [ ]:
run = cfg['run']
print(f"Analysis: {cfg['analysis']['name']}")
print(f"Target ratio: {run['numerator_target']}/{run['denominator_target']} at {run['angle_deg']} deg")
print(f"Data cut: {cfg['cuts']['data_electron']['expression'].strip()}")
print(f"SIMC cut: {cfg['cuts']['simc_acceptance']['expression'].strip()}")
print('\nCorrections:')
for name in cfg['corrections']:
    print(f"- {name}")
print('\nModules:')
for name, module in cfg['modules'].items():
    state = 'enabled' if module.get('enabled', False) else 'disabled'
    print(f"- {name}: {state}")